# 05 Silver — `fact_urna_event`

Transforma a camada Bronze `fact_urna_logs` (mensagens de texto dos logs de urna) na tabela Silver `fact_urna_event`, classificando semanticamente cada linha de log contra o catálogo declarativo de eventos (`event_catalog.yml`) e descartando todo texto livre e PII.

### Objetivos:
- Ler todos os Parquet Bronze particionados em `election_year/turn/uf`.
- Classificar cada linha com `EventCatalog.classify()` (allowlist semântica, zero-PII).
- Produzir Silver Parquet com 16 colunas tipadas (incluindo `event_code`, `classification_status`, `biometric_attempt_num`, `parser_version`).
- Reconciliar contagem Bronze == Silver.
- Emitir relatório de cobertura com `classification_status` breakdown e `UNKNOWN_EVENT %`.

In [ ]:
import importlib
import importlib.util
import inspect
import os
import shutil
import stat
import subprocess
import sys
from pathlib import Path

_REQUIRED_PACKAGES = (
    ("kagglehub", "kagglehub>=0.4.0"),
    ("polars", "polars"),
    ("pyarrow", "pyarrow"),
    ("duckdb", "duckdb"),
    ("yaml", "pyyaml"),
)
if globals().get("__name__") == "__main__" and not os.getenv("PYTEST_CURRENT_TEST"):
    for _mod_name, _pip_name in _REQUIRED_PACKAGES:
        if importlib.util.find_spec(_mod_name) is None:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "-U", _pip_name]
            )
            importlib.invalidate_caches()
    import kagglehub as _kh_check

    if "output_dir" not in inspect.signature(_kh_check.dataset_download).parameters:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-U", "kagglehub>=0.4.0"]
        )
        importlib.invalidate_caches()
        sys.modules.pop("kagglehub", None)
        import kagglehub as _kh_check

# Contrato único de estado e ciclo de vida via KaggleHub SDK (Kaggle e Google Colab)
DATASET_HANDLE = os.getenv("KAGGLE_DATASET_HANDLE", "daniellmattos/tse-eleicoes-dados-v2")
DATASET_PATH = os.getenv("DATASET_PATH", "/content/drive/MyDrive/tse-eleicoes-dados/data")
STAGING_PATH = os.getenv("STAGING_PATH", "/tmp/tse-eleicoes-staging")
ALLOW_COLD_START = False


def is_directory_empty(path: Path) -> bool:
    """Verifica se DATASET_PATH está efetivamente vazio (ignorando arquivos ocultos de sistema)."""
    if not path.exists():
        return True
    return not any(item for item in path.iterdir() if not item.name.startswith("."))


def ensure_writable_tree(root: Path) -> None:
    """Garante permissão de escrita em árvores copiadas de mounts somente-leitura."""
    if not root.exists():
        return
    root.chmod(stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
    for dirpath, dirnames, filenames in os.walk(root):
        base = Path(dirpath)
        for dname in dirnames:
            dp = base / dname
            if not dp.is_symlink():
                dp.chmod(stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
        for fname in filenames:
            fp = base / fname
            if not fp.is_symlink():
                fp.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IRGRP | stat.S_IROTH)


def is_dataset_not_found_error(exc: BaseException) -> bool:
    """Detecta erros de dataset inexistente (404 / Not Found) para permitir reconstrução do zero."""
    msg = f"{type(exc).__name__}: {exc}".lower()
    not_found_markers = (
        "404",
        "not found",
        "does not exist",
        "no dataset found",
        "resource not found",
        "403",
    )
    return any(marker in msg for marker in not_found_markers)


def ensure_kaggle_auth() -> None:
    """Autentica via KaggleHub SDK sem travar execuções headless (batch kernels / CI)."""
    import kagglehub

    has_env_creds = bool(
        os.getenv("KAGGLE_API_TOKEN")
        or (os.getenv("KAGGLE_USERNAME") and os.getenv("KAGGLE_KEY"))
        or (Path.home() / ".kaggle" / "kaggle.json").exists()
        or Path("/root/.config/kaggle/kaggle.json").exists()
    )
    if has_env_creds:
        print("[AUTH] Credenciais do Kaggle detectadas no ambiente.")
        return
    if sys.stdin is not None and sys.stdin.isatty():
        print("Iniciando autenticação manual no Kaggle SDK...")
        kagglehub.login()
    else:
        print(
            "[AUTH] Execução não-interativa detectada; pulando prompt interativo kagglehub.login()."
        )


def bootstrap_dataset(
    handle: str, target_dir: Path, allow_cold_start: bool = False
) -> Path:
    """Se DATASET_PATH estiver vazio, baixa a última versão do dataset do Kaggle ou reconstrói do zero se não existir."""
    target_dir.mkdir(parents=True, exist_ok=True)
    if not is_directory_empty(target_dir):
        print(f"[OK] DATASET_PATH ({target_dir}) já contém dados locais. Pulando download.")
        return target_dir

    print(
        f"[BOOTSTRAP] DATASET_PATH ({target_dir}) está vazio. Baixando '{handle}' via Kaggle SDK..."
    )
    try:
        import kagglehub

        try:
            downloaded_path = Path(
                kagglehub.dataset_download(
                    handle,
                    output_dir=str(target_dir),
                    force_download=True,
                )
            )
        except TypeError as type_err:
            if "output_dir" not in str(type_err):
                raise
            downloaded_path = Path(
                kagglehub.dataset_download(
                    handle,
                    force_download=True,
                )
            )

        if downloaded_path.resolve() != target_dir.resolve() and downloaded_path.exists():
            print(
                f"[SYNC] Copiando arquivos de {downloaded_path} (read-only mount) para {target_dir}..."
            )
            shutil.copytree(downloaded_path, target_dir, dirs_exist_ok=True)
            ensure_writable_tree(target_dir)

        print(f"[OK] Dataset populado com sucesso em {target_dir}.")
    except Exception as exc:
        if allow_cold_start or is_dataset_not_found_error(exc):
            print(
                f"[COLD-START] Dataset '{handle}' não encontrado ou ainda não publicado ({exc}). "
                f"Iniciando reconstrução do zero em {target_dir}."
            )
        else:
            raise RuntimeError(
                f"Falha ao popular DATASET_PATH ({target_dir}) a partir do Kaggle ('{handle}'). "
                "Verifique sua autenticação e conexão de rede."
            ) from exc

    return target_dir


if globals().get("__name__") == "__main__" and not os.getenv("PYTEST_CURRENT_TEST"):
    _drive_mount_point = Path("/content/drive")
    if not (_drive_mount_point / "MyDrive").exists():
        try:
            from google.colab import drive

            drive.mount(str(_drive_mount_point))
        except Exception as _exc:
            print(f"[BOOT] Headless/CLI execution detected while checking Drive ({_exc}).")

    _token_cache_file = Path("/root/.config/kaggle/colab_token.env")
    if not os.environ.get("KAGGLE_API_TOKEN"):
        if _token_cache_file.exists():
            os.environ["KAGGLE_API_TOKEN"] = _token_cache_file.read_text(encoding="utf-8").strip()
        else:
            try:
                from google.colab import userdata

                _token = userdata.get("KAGGLE_API_TOKEN")
                if _token:
                    os.environ["KAGGLE_API_TOKEN"] = _token.strip()
                    _token_cache_file.parent.mkdir(parents=True, exist_ok=True)
                    _token_cache_file.write_text(_token.strip(), encoding="utf-8")
                    _token_cache_file.chmod(0o600)
            except Exception as _exc:
                print(
                    f"[BOOT] userdata unavailable in CLI mode ({_exc}); "
                    "relying on environment variables."
                )
    Path(DATASET_PATH).mkdir(parents=True, exist_ok=True)
    Path(STAGING_PATH).mkdir(parents=True, exist_ok=True)
    ensure_kaggle_auth()
    bootstrap_dataset(
        DATASET_HANDLE, Path(DATASET_PATH), allow_cold_start=ALLOW_COLD_START
    )

import datetime
import json
import re

import polars as pl
import yaml

ELECTION_YEAR = 2026
TURNS = [1, 2]
ALL_UFS = [
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA",
    "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN",
    "RO", "RR", "RS", "SC", "SE", "SP", "TO", "ZZ",
]
PARSER_VERSION = "1.0.0"


def find_table_dir(table_name, roots=(Path(DATASET_PATH),), max_depth=7):
    """Localiza <table_name>/ com partições election_year=* sob DATASET_PATH."""
    for root in roots:
        if not root.exists():
            continue
        base = len(root.parts)
        for dirpath, dirnames, _ in os.walk(root):
            p = Path(dirpath)
            if len(p.parts) - base >= max_depth:
                dirnames[:] = []
                continue
            if p.name == table_name and any(d.startswith("election_year=") for d in dirnames):
                return p
    return None


BRONZE_BASE = find_table_dir("fact_urna_logs")
if BRONZE_BASE is None:
    raise FileNotFoundError(
        f"Tabela Bronze fact_urna_logs não encontrada em {DATASET_PATH}. "
        f"Execute a Fase 03 ou verifique o dataset {DATASET_HANDLE}."
    )
print(f"BRONZE_BASE resolvido: {BRONZE_BASE}")
SILVER_OUT = Path(DATASET_PATH) / "silver" / "fact_urna_event"
SILVER_OUT.mkdir(parents=True, exist_ok=True)

print(f"Setup concluído. Polars {pl.__version__}, {len(ALL_UFS)} UFs, {len(TURNS)} turno(s).")

In [ ]:
# ---------------------------------------------------------------------------
# Catálogo declarativo de eventos — embutido inline (zero-PII, allowlist)
# ---------------------------------------------------------------------------

EVENT_CATALOG_YAML = r"""
schema_version: "1.0"
fallback_event_code: "UNKNOWN_EVENT"
malformed_event_code: "MALFORMED_LINE"
allowed_extractors:
  - "biometric_attempt_num"

required_event_codes:
  - "LOGD_STARTED"
  - "SECTION_OPENED"
  - "VOTER_ENABLED"
  - "BIOMETRIC_READ_ATTEMPT"
  - "BIOMETRIC_RECOGNIZED"
  - "BIOMETRIC_NOT_RECOGNIZED"
  - "NO_BIOMETRIC_REGISTERED"
  - "MANUAL_ENABLE_PATH"
  - "INACTIVITY_TIMEOUT_45S"
  - "INVALID_KEY_PRESSED"
  - "CARGO_VOTE_RECORDED"
  - "VOTER_VOTE_COMPUTED"
  - "VOTING_SUSPENDED"
  - "VOTING_RESUMED"
  - "CONTINGENCY_EVENT"
  - "SECTION_CLOSED"

patterns:
  - pattern_id: "logd_started_main"
    event_code: "LOGD_STARTED"
    applications: ["*"]
    regex: "(?i)(?:in[ií]cio\\s+das\\s+opera[cç][oõ]es\\s+do\\s+logd|logd\\s+iniciado)"
    extractors: {}
    description: "Inicialização do serviço de log (logd) da urna eletrônica."

  - pattern_id: "section_opened_main"
    event_code: "SECTION_OPENED"
    applications: ["VOTA", "GAP", "*"]
    regex: "(?i)(?:urna\\s+pronta\\s+para\\s+receber\\s+votos|in[ií]cio\\s+da\\s+vota[cç][aã]o|vota[cç][aã]o\\s+iniciada|abertura\\s+da\\s+se[cç][aã]o|emiss[aã]o\\s+d[ae]\\s+zer[eé]sima)"
    extractors: {}
    description: "Abertura da seção eleitoral, emissão da Zerésima ou início da votação."

  - pattern_id: "voter_enabled_main"
    event_code: "VOTER_ENABLED"
    applications: ["VOTA", "*"]
    regex: "(?i)^(?=.*\\beleitor\\s+(?:foi\\s+)?habilitado\\b)(?!.*\\b(?:sem\\s+biometria|manual)\\b).*$"
    extractors: {}
    description: "Habilitação do eleitor para iniciar a sessão de votação."

  - pattern_id: "biometric_read_attempt_numbered"
    event_code: "BIOMETRIC_READ_ATTEMPT"
    applications: ["VOTA", "*"]
    regex: "(?i)^(?!(?:.*\\b(?:mes[aá]rio|presidente|autoteste|teste\\s+de\\s+sensor)\\b))(?:.*(?:tentativa\\s+de\\s+(?:reconhecimento|leitura|identifica[cç][aã]o)\\s+biom[eé]tric[oa]|solicita(?:ndo)?\\s+digital[.:;]?\\s*tentativa|capturada\\s+a\\s+digital[.:;]?\\s*tentativa|n[uú]mero\\s+de\\s+tentativas\\s+de\\s+reconhecimento\\s+do\\s+dedo(?:[.:;]?\\s*tentativa)?)\\s*(?:n[uúº°.]*\\s*)?[:=\\-]?\\s*[\\[\\(]?(?P<biometric_attempt_num>\\d{1,3})[\\]\\)]?(?:\\b|$))"
    extractors:
      biometric_attempt_num: "biometric_attempt_num"
    description: "Tentativa numerada de leitura ou reconhecimento biométrico da digital do eleitor."

  - pattern_id: "biometric_read_attempt_generic"
    event_code: "BIOMETRIC_READ_ATTEMPT"
    applications: ["VOTA", "*"]
    regex: "(?i)^(?!(?:.*\\b(?:mes[aá]rio|presidente|autoteste|teste\\s+de\\s+sensor)\\b))(?:.*(?:tentativa\\s+de\\s+(?:reconhecimento|leitura|identifica[cç][aã]o)\\s+biom[eé]tric[oa]|solicita(?:[cç][aã]o\\s+de)?\\s+(?:digital|biometria)(?:\\s+do\\s+eleitor)?|aguardando\\s+(?:leitura\\s+biom[eé]trica|digital\\s+do\\s+eleitor)|n[uú]mero\\s+de\\s+tentativas\\s+de\\s+reconhecimento\\s+do\\s+dedo))"
    extractors: {}
    description: "Solicitação ou tentativa de leitura biométrica do eleitor sem número explícito na mensagem."

  - pattern_id: "biometric_not_recognized_main"
    event_code: "BIOMETRIC_NOT_RECOGNIZED"
    applications: ["VOTA", "*"]
    regex: "(?i)^(?!(?:.*\\b(?:mes[aá]rio|presidente|autoteste|teste\\s+de\\s+sensor)\\b))(?:.*(?:(?:digital|biometria|dedo)(?:\\s+do\\s+eleitor|\\s+capturad[oa])?\\s+n[aã]o\\s+(?:foi\\s+)?reconhecid[oa]|timeout\\s+de\\s+reconhecimento\\s+do\\s+dedo|falha\\s+no\\s+reconhecimento\\s+biom[eé]trico|impress[aã]o\\s+digital\\s+n[aã]o\\s+confere))"
    extractors: {}
    description: "Falha, timeout ou não reconhecimento da impressão digital do eleitor."

  - pattern_id: "biometric_recognized_main"
    event_code: "BIOMETRIC_RECOGNIZED"
    applications: ["VOTA", "*"]
    regex: "(?i)^(?!(?:.*\\b(?:mes[aá]rio|presidente|autoteste|teste\\s+de\\s+sensor)\\b))(?:.*(?:(?<!n[aã]o\\s)(?<!n[aã]o\\sfoi\\s)\\b(?:biometria|digital|dedo)(?:\\s+do\\s+eleitor)?\\s+reconhecid[oa]\\b|tipo\\s+de\\s+habilita[cç][aã]o\\s+do\\s+eleitor\\s*\\[?biom[eé]trica\\]?|reconhecimento\\s+biom[eé]trico\\s+(?:efetuado\\s+com\\s+sucesso|positivo|ok)|identifica[cç][aã]o\\s+biom[eé]trica\\s+positiva|eleitor\\s+identificado\\s+por\\s+biometria))"
    extractors: {}
    description: "Reconhecimento biométrico bem-sucedido da impressão digital do eleitor."

  - pattern_id: "no_biometric_registered_main"
    event_code: "NO_BIOMETRIC_REGISTERED"
    applications: ["VOTA", "*"]
    regex: "(?i)^(?!(?:.*\\b(?:mes[aá]rio|presidente)\\b))(?:.*(?:n[aã]o\\s+h[aá]\\s+biometria(?:\\s+cadastrada)?|eleitor\\s+(?:n[aã]o\\s+possui|sem)\\s+biometria(?:\\s+cadastrada)?|aus[eê]ncia\\s+de\\s+biometria\\s+no\\s+cadastro|habilita[cç][aã]o\\s+de\\s+eleitor\\s+sem\\s+biometria|ano\\s+de\\s+nascimento\\s+digitado\\s+[eé]\\s+igual\\s+ao\\s+do\\s+cadastro))"
    extractors: {}
    description: "Eleitor sem biometria cadastrada no banco de dados da seção."

  - pattern_id: "manual_enable_path_main"
    event_code: "MANUAL_ENABLE_PATH"
    applications: ["VOTA", "*"]
    regex: "(?i)(?:habilita[cç][aã]o\\s+manual(?:\\s+pelo\\s+mes[aá]rio)?|libera[cç][aã]o\\s+manual(?:\\s+pelo\\s+mes[aá]rio)?|eleitor\\s+habilitado\\s+(?:manualmente|sem\\s+biometria)|habilitado\\s+sem\\s+biometria\\s+ap[oó]s\\s+tentativas|identifica[cç][aã]o\\s+manual\\s+do\\s+eleitor|c[oó]digo\\s+de\\s+habilita[cç][aã]o\\s+digitado\\s+pelo\\s+mes[aá]rio|solicita[cç][aã]o\\s+de\\s+dado\\s+pessoal\\s+do\\s+eleitor\\s+para\\s+habilita[cç][aã]o\\s+manual|mes[aá]rio\\s+\\d+\\s+habilitou\\s+o\\s+eleitor)"
    extractors: {}
    description: "Habilitação manual pelo mesário após tentativas biométricas malsucedidas."

  - pattern_id: "inactivity_timeout_45s_main"
    event_code: "INACTIVITY_TIMEOUT_45S"
    applications: ["VOTA", "*"]
    regex: "(?i)(?:eleitor\\s+sem\\s+atividade\\s+por\\s+45\\s*s(?:egundos)?|sem\\s+atividade\\s+por\\s+45\\s*s(?:egundos)?|tempo\\s+limite\\s+de\\s+inatividade|timeout\\s+de\\s+inatividade|inatividade\\s+(?:de|por)\\s+45\\s*s(?:egundos)?|cancelamento\\s+por\\s+inatividade(?:\\s+de\\s+45\\s*s(?:egundos)?)?)"
    extractors: {}
    description: "Alerta ou cancelamento por inatividade de 45 segundos durante a sessão."

  - pattern_id: "invalid_key_pressed_main"
    event_code: "INVALID_KEY_PRESSED"
    applications: ["VOTA", "*"]
    regex: "(?i)(?:acionamento\\s+de\\s+)?tecla\\s+(?:indevida|inv[aá]lida)(?:\\s+pressionada)?"
    extractors: {}
    description: "Acionamento de tecla indevida ou inválida durante a habilitação ou votação."

  - pattern_id: "cargo_vote_recorded_main"
    event_code: "CARGO_VOTE_RECORDED"
    applications: ["VOTA", "*"]
    regex: "(?i)(?:voto\\s+(?:confirmado|registrado)\\s+para\\b|confirma[cç][aã]o\\s+de\\s+voto\\s+no\\s+cargo)"
    extractors: {}
    description: "Confirmação de voto em um cargo individual."

  - pattern_id: "voter_vote_computed_main"
    event_code: "VOTER_VOTE_COMPUTED"
    applications: ["VOTA", "*"]
    regex: "(?i)(?:o\\s+)?voto\\s+do\\s+eleitor\\s+(?:foi\\s+)?computado"
    extractors: {}
    description: "Conclusão e computação final do voto do eleitor na seção."

  - pattern_id: "voting_suspended_main"
    event_code: "VOTING_SUSPENDED"
    applications: ["VOTA", "GAP", "*"]
    regex: "(?i)(?:vota[cç][aã]o\\s+suspensa|suspens[aã]o\\s+(?:tempor[aá]ria\\s+|parcial\\s+|total\\s+)?da\\s+vota[cç][aã]o|eleitor\\s+suspenso\\s+pelo\\s+mes[aá]rio)"
    extractors: {}
    description: "Suspensão temporária, parcial ou total da votação na seção eleitoral."

  - pattern_id: "voting_resumed_main"
    event_code: "VOTING_RESUMED"
    applications: ["VOTA", "GAP", "*"]
    regex: "(?i)(?:vota[cç][aã]o\\s+retomada|retomada\\s+da\\s+vota[cç][aã]o)"
    extractors: {}
    description: "Retomada da votação após suspensão ou procedimento técnico."

  - pattern_id: "contingency_event_main"
    event_code: "CONTINGENCY_EVENT"
    applications: ["VOTA", "GAP", "RED", "scue", "adm", "*"]
    regex: "(?i)(?:modo\\s+de\\s+conting[eê]ncia|substitui[cç][aã]o\\s+de\\s+urna|recupera[cç][aã]o\\s+de\\s+dados\\s+de\\s+conting[eê]ncia|carga\\s+de\\s+conting[eê]ncia|in[ií]cio\\s+de\\s+conting[eê]ncia)"
    extractors: {}
    description: "Operação técnica de contingência, carga ou substituição de urna."

  - pattern_id: "section_closed_main"
    event_code: "SECTION_CLOSED"
    applications: ["VOTA", "GAP", "RED", "*"]
    regex: "(?i)(?:encerramento\\s+da\\s+(?:vota[cç][aã]o|se[cç][aã]o)|vota[cç][aã]o\\s+encerrada|se[cç][aã]o\\s+encerrada|emiss[aã]o\\s+do\\s+boletim\\s+de\\s+urna|gerando\\s+arquivo\\s+de\\s+resultado\\s*\\[imgbu\\.dat\\]\\s*\\+\\s*\\[t[eé]rmino\\])"
    extractors: {}
    description: "Encerramento oficial da votação na seção eleitoral e emissão do BU."
"""

# ---------------------------------------------------------------------------
# Dataclasses e funções inline (sem import tse_analysis)
# ---------------------------------------------------------------------------
from dataclasses import dataclass
from dataclasses import field as dc_field
from typing import Any

UNKNOWN_EVENT_CODE: str = "UNKNOWN_EVENT"
MALFORMED_LINE_CODE: str = "MALFORMED_LINE"
ALLOWED_EXTRACTOR_FIELDS: tuple = ("biometric_attempt_num",)
FORBIDDEN_EXTRACTOR_TOKENS: tuple = (
    "cpf", "titulo", "eleitor", "mesario", "message",
    "text", "mac", "voto", "votavel", "candidato",
)


class EventCatalogValidationError(ValueError):
    pass


@dataclass(frozen=True)
class ClassificationResult:
    event_code: str
    biometric_attempt_num: int | None = None
    pattern_id: str | None = None


@dataclass(frozen=True)
class EventPattern:
    pattern_id: str
    event_code: str
    applications: tuple
    regex: str
    compiled_regex: re.Pattern
    extractors: dict = dc_field(default_factory=dict)
    description: str = ""

    def matches_application(self, application: str) -> bool:
        if "*" in self.applications:
            return True
        normalized_app = application.strip().upper()
        return any(app.upper() == normalized_app for app in self.applications)


@dataclass(frozen=True)
class EventCatalog:
    schema_version: str
    fallback_event_code: str
    malformed_event_code: str
    patterns: tuple

    @property
    def event_codes(self) -> frozenset:
        return frozenset(p.event_code for p in self.patterns)

    def classify(self, message_text: str, application: str = "VOTA") -> ClassificationResult:
        normalized_msg = message_text.strip()
        if not normalized_msg:
            return ClassificationResult(
                event_code=self.fallback_event_code,
                biometric_attempt_num=None,
                pattern_id=None,
            )
        for pattern in self.patterns:
            if not pattern.matches_application(application):
                continue
            match = pattern.compiled_regex.search(normalized_msg)
            if match is None:
                continue
            attempt_num: int | None = None
            group_name = pattern.extractors.get("biometric_attempt_num")
            if group_name:
                raw_val = match.groupdict().get(group_name)
                if raw_val is not None:
                    try:
                        parsed_int = int(raw_val)
                        if 1 <= parsed_int <= 127:
                            attempt_num = parsed_int
                    except ValueError:
                        attempt_num = None
            return ClassificationResult(
                event_code=pattern.event_code,
                biometric_attempt_num=attempt_num,
                pattern_id=pattern.pattern_id,
            )
        return ClassificationResult(
            event_code=self.fallback_event_code,
            biometric_attempt_num=None,
            pattern_id=None,
        )


def _validate_and_build_pattern(raw: Any, seen_pattern_ids: set) -> EventPattern:
    if not isinstance(raw, dict):
        raise EventCatalogValidationError("Cada item em 'patterns' deve ser um objeto YAML.")
    for req_field in ("pattern_id", "event_code", "applications", "regex"):
        if req_field not in raw or raw[req_field] is None:
            raise EventCatalogValidationError(
                f"Regra de evento inválida: campo obrigatório ausente '{req_field}'."
            )
    pattern_id = str(raw["pattern_id"]).strip()
    if not pattern_id:
        raise EventCatalogValidationError("Campo 'pattern_id' não pode ser vazio.")
    if pattern_id in seen_pattern_ids:
        raise EventCatalogValidationError(f"pattern_id duplicado no catálogo: '{pattern_id}'.")
    seen_pattern_ids.add(pattern_id)

    event_code = str(raw["event_code"]).strip()
    if not event_code or not re.fullmatch(r"[A-Z][A-Z0-9_]*", event_code):
        raise EventCatalogValidationError(
            f"Regra '{pattern_id}' possui event_code inválido '{event_code}'. Use UPPER_SNAKE_CASE."
        )
    apps_raw = raw["applications"]
    if not isinstance(apps_raw, list) or not apps_raw:
        raise EventCatalogValidationError(
            f"Regra '{pattern_id}' deve definir 'applications' como lista não-vazia."
        )
    applications = tuple(str(app).strip() for app in apps_raw if str(app).strip())

    regex_str = str(raw["regex"])
    if not regex_str.strip():
        raise EventCatalogValidationError(f"Regra '{pattern_id}' possui regex vazia.")
    try:
        compiled = re.compile(regex_str)
    except re.error as exc:
        raise EventCatalogValidationError(
            f"Expressão regular inválida na regra '{pattern_id}': {exc}"
        ) from exc

    for group_name in compiled.groupindex:
        lowered_group = group_name.lower()
        if any(token in lowered_group for token in FORBIDDEN_EXTRACTOR_TOKENS):
            raise EventCatalogValidationError(
                f"Regra '{pattern_id}' declara grupo regex proibido '{group_name}' "
                "(violação de política zero-PII/zero-texto-livre)."
            )

    extractors_raw = raw.get("extractors") or {}
    if not isinstance(extractors_raw, dict):
        raise EventCatalogValidationError(
            f"Campo 'extractors' na regra '{pattern_id}' deve ser um mapeamento."
        )
    extractors: dict = {}
    for target_field, source_group in extractors_raw.items():
        field_str = str(target_field).strip()
        group_str = str(source_group).strip()
        lowered_field = field_str.lower()
        if (
            any(token in lowered_field for token in FORBIDDEN_EXTRACTOR_TOKENS)
            or field_str not in ALLOWED_EXTRACTOR_FIELDS
        ):
            raise EventCatalogValidationError(
                f"Regra '{pattern_id}' tenta extrair campo não permitido '{field_str}'."
            )
        if group_str not in compiled.groupindex:
            raise EventCatalogValidationError(
                f"Regra '{pattern_id}' referencia grupo regex inexistente '{group_str}'."
            )
        extractors[field_str] = group_str

    return EventPattern(
        pattern_id=pattern_id,
        event_code=event_code,
        applications=applications,
        regex=regex_str,
        compiled_regex=compiled,
        extractors=extractors,
        description=str(raw.get("description", "")).strip(),
    )


def load_event_catalog_from_string(yaml_str: str) -> EventCatalog:
    """Carrega, compila e valida o catálogo a partir de uma string YAML inline."""
    payload = yaml.safe_load(yaml_str)
    if not isinstance(payload, dict):
        raise EventCatalogValidationError("O conteúdo YAML deve ser um objeto raiz.")
    raw_patterns = payload.get("patterns")
    if not isinstance(raw_patterns, list) or not raw_patterns:
        raise EventCatalogValidationError(
            "O catálogo deve conter uma lista não-vazia em 'patterns'."
        )
    seen_ids: set = set()
    patterns = tuple(_validate_and_build_pattern(item, seen_ids) for item in raw_patterns)
    return EventCatalog(
        schema_version=str(payload.get("schema_version", "1.0")),
        fallback_event_code=str(payload.get("fallback_event_code", UNKNOWN_EVENT_CODE)),
        malformed_event_code=str(payload.get("malformed_event_code", MALFORMED_LINE_CODE)),
        patterns=patterns,
    )


CATALOG = load_event_catalog_from_string(EVENT_CATALOG_YAML)
print(f"Catálogo carregado: {len(CATALOG.patterns)} padrões")

In [ ]:
# ---------------------------------------------------------------------------
# Silver schema definition
# ---------------------------------------------------------------------------
SILVER_SCHEMA = pl.Schema({
    "election_year": pl.Int16,
    "turn": pl.Int8,
    "uf": pl.String,
    "municipio_code": pl.Int32,
    "zone_num": pl.Int16,
    "section_num": pl.Int32,
    "urna_serial": pl.String,
    "file_seq": pl.Int64,
    "raw_timestamp": pl.Datetime("us"),
    "has_timestamp_regression": pl.Boolean,
    "application": pl.String,
    "log_level": pl.String,
    "event_code": pl.String,
    "biometric_attempt_num": pl.Int8,
    "classification_status": pl.String,
    "parser_version": pl.String,
})

print("Silver schema:")
for col_name, col_type in SILVER_SCHEMA.items():
    print(f"  {col_name}: {col_type}")

In [ ]:
# ---------------------------------------------------------------------------
# classify_row helper
# ---------------------------------------------------------------------------
def classify_row(message_text: str, application: str, catalog: EventCatalog) -> dict:
    """Classifica uma linha de log e retorna dict com event_code, biometric_attempt_num, classification_status."""
    result = catalog.classify(message_text, application)
    is_classified = result.event_code not in (
        catalog.fallback_event_code,
        catalog.malformed_event_code,
    )
    return {
        "event_code": result.event_code,
        "biometric_attempt_num": result.biometric_attempt_num,
        "classification_status": "CLASSIFIED" if is_classified else "UNKNOWN",
    }


# Quick smoke test
test_result = classify_row("Voto do eleitor computado", "VOTA", CATALOG)
assert test_result["event_code"] == "VOTER_VOTE_COMPUTED", f"Smoke test falhou: {test_result}"
print(f"classify_row smoke test OK: {test_result}")

In [ ]:
# ---------------------------------------------------------------------------
# Main loop: Bronze -> classify -> Silver
# ---------------------------------------------------------------------------
import time

bronze_total = 0
silver_total = 0
skipped_partitions = []

t0 = time.time()

for turn in TURNS:
    for uf in ALL_UFS:
        partition_glob = BRONZE_BASE / f"election_year={ELECTION_YEAR}" / f"turn={turn}" / f"uf={uf}" / "*.parquet"
        parquet_files = sorted(partition_glob.parent.glob("*.parquet")) if partition_glob.parent.exists() else []

        if not parquet_files:
            skipped_partitions.append(f"turn={turn}/uf={uf}")
            continue

        local_part_stage = (
            Path(STAGING_PATH)
            / "silver_fact_urna_event"
            / f"election_year={ELECTION_YEAR}"
            / f"turn={turn}"
            / f"uf={uf}"
        )
        if local_part_stage.exists():
            shutil.rmtree(local_part_stage, ignore_errors=True)
        local_part_stage.mkdir(parents=True, exist_ok=True)

        part_bronze_rows = 0
        part_silver_rows = 0
        staged_files: list[Path] = []
        try:
            for shard_idx, pq_file in enumerate(parquet_files):
                bronze_df = pl.read_parquet(pq_file)
                n_bronze = len(bronze_df)
                part_bronze_rows += n_bronze

                messages = bronze_df["message_text"].to_list()
                if "application" in bronze_df.columns:
                    applications = bronze_df["application"].to_list()
                else:
                    applications = ["VOTA"] * n_bronze

                event_codes = []
                biometric_nums = []
                classification_statuses = []

                for msg, app in zip(messages, applications):
                    classified = classify_row(
                        str(msg) if msg is not None else "",
                        str(app) if app else "VOTA",
                        CATALOG,
                    )
                    event_codes.append(classified["event_code"])
                    biometric_nums.append(classified["biometric_attempt_num"])
                    classification_statuses.append(classified["classification_status"])

                bronze_cols = bronze_df.columns

                def _get_col(name: str, default_val, dtype, aliases=(), _df=bronze_df, _cols=bronze_cols, _n=n_bronze):
                    # Bronze usa cd_municipio/nr_zona/nr_secao; Silver usa municipio_code/zone_num/section_num.
                    for src_name in (name, *aliases):
                        if src_name in _cols:
                            try:
                                return _df[src_name].cast(dtype).alias(name)
                            except Exception:
                                break
                    return pl.Series(name, [default_val] * _n, dtype=dtype)

                silver_df = pl.DataFrame({
                    "election_year": _get_col("election_year", ELECTION_YEAR, pl.Int16),
                    "turn": _get_col("turn", turn, pl.Int8),
                    "uf": _get_col("uf", uf, pl.String),
                    "municipio_code": _get_col("municipio_code", None, pl.Int32, aliases=("cd_municipio",)),
                    "zone_num": _get_col("zone_num", None, pl.Int16, aliases=("nr_zona",)),
                    "section_num": _get_col("section_num", None, pl.Int32, aliases=("nr_secao",)),
                    "urna_serial": _get_col("urna_serial", None, pl.String),
                    "file_seq": _get_col("file_seq", None, pl.Int64),
                    "raw_timestamp": _get_col("raw_timestamp", None, pl.Datetime("us")),
                    "has_timestamp_regression": _get_col("has_timestamp_regression", False, pl.Boolean),
                    "application": pl.Series("application", applications, dtype=pl.String),
                    "log_level": _get_col("log_level", None, pl.String),
                    "event_code": pl.Series("event_code", event_codes, dtype=pl.String),
                    "biometric_attempt_num": pl.Series("biometric_attempt_num", biometric_nums, dtype=pl.Int8),
                    "classification_status": pl.Series("classification_status", classification_statuses, dtype=pl.String),
                    "parser_version": pl.Series("parser_version", [PARSER_VERSION] * n_bronze, dtype=pl.String),
                })

                local_out = local_part_stage / f"part-{shard_idx}.parquet"
                silver_df.write_parquet(local_out, compression="zstd", compression_level=5)
                staged_files.append(local_out)
                part_silver_rows += len(silver_df)
                del bronze_df, silver_df, messages, applications, event_codes, biometric_nums, classification_statuses

            # Drive FUSE safety: limpa partição no destino antes de copiar shards finalizados
            out_dir = SILVER_OUT / f"election_year={ELECTION_YEAR}" / f"turn={turn}" / f"uf={uf}"
            if out_dir.exists():
                shutil.rmtree(out_dir)
            out_dir.mkdir(parents=True, exist_ok=True)
            for local_out in staged_files:
                dest_out = out_dir / local_out.name
                shutil.copyfile(local_out, dest_out)
                if dest_out.stat().st_size != local_out.stat().st_size:
                    raise OSError(f"Tamanho divergente após cópia para {dest_out}")
        finally:
            shutil.rmtree(local_part_stage, ignore_errors=True)

        bronze_total += part_bronze_rows
        silver_total += part_silver_rows
        print(
            f"  turn={turn} uf={uf}: {part_bronze_rows:,} Bronze -> {part_silver_rows:,} Silver rows "
            f"({len(staged_files)} part(s)) -> {out_dir}"
        )

elapsed = time.time() - t0
print(f"\nLoop concluído em {elapsed:.1f}s")
print(f"Partições ignoradas (sem dados Bronze): {len(skipped_partitions)}")
print(f"Bronze total: {bronze_total:,} | Silver total: {silver_total:,}")

In [ ]:
# ---------------------------------------------------------------------------
# Reconciliação: Bronze total == Silver total
# ---------------------------------------------------------------------------
if bronze_total == 0:
    raise RuntimeError("Nenhuma linha Bronze foi lida: a Silver ficaria vazia. Verifique o dataset de entrada.")
print(f"Bronze rows ingested : {bronze_total:,}")
print(f"Silver rows produced : {silver_total:,}")

if bronze_total != silver_total:
    raise AssertionError(
        f"RECONCILIATION FAILED: bronze_total={bronze_total:,} != silver_total={silver_total:,}. "
        "Verifique os logs acima para identificar partições com discrepância."
    )

print(f"\nReconciliação OK: {bronze_total:,} linhas Bronze == {silver_total:,} linhas Silver.")

In [ ]:
# ---------------------------------------------------------------------------
# Coverage report + manifest
# ---------------------------------------------------------------------------
silver_all_files = sorted(SILVER_OUT.rglob("*.parquet"))
print(f"Arquivos Silver gerados: {len(silver_all_files)}")

if silver_all_files:
    silver_lf = pl.scan_parquet([str(p) for p in silver_all_files])

    print("\n--- event_code value counts ---")
    event_code_counts = (
        silver_lf
        .group_by("event_code")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )
    print(event_code_counts)

    print("\n--- classification_status breakdown ---")
    status_counts = (
        silver_lf
        .group_by("classification_status")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )
    print(status_counts)

    total_silver = int(silver_lf.select(pl.len()).collect().item())
    unknown_count = int(
        silver_lf
        .filter(pl.col("event_code") == "UNKNOWN_EVENT")
        .select(pl.len())
        .collect()
        .item()
    )
    unknown_pct = 100.0 * unknown_count / total_silver if total_silver > 0 else 0.0
    print(f"\nUNKNOWN_EVENT: {unknown_count:,} / {total_silver:,} = {unknown_pct:.2f}%")

    # Manifest
    manifest = {
        "notebook": "05_silver_fact_urna_event",
        "parser_version": PARSER_VERSION,
        "election_year": ELECTION_YEAR,
        "bronze_total_rows": bronze_total,
        "silver_total_rows": silver_total,
        "silver_parquet_files": len(silver_all_files),
        "unknown_event_count": unknown_count,
        "unknown_event_pct": round(unknown_pct, 4),
        "classification_status_counts": {
            row["classification_status"]: row["count"]
            for row in status_counts.to_dicts()
        },
        "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",
    }
    print("\n--- Manifest ---")
    print(json.dumps(manifest, indent=2, ensure_ascii=False))
else:
    print("Nenhum arquivo Silver encontrado — verifique se o Bronze contém dados para os turnos/UFs configurados.")

# Validação de higiene pré-upload e publicação de nova versão via KaggleHub SDK
PHASE_ID = "05_silver_fact_urna_event"
PHASE_SUMMARY = "Camada Silver: classificação semântica de eventos em fact_urna_event (Parquet ZSTD)"
forbidden_extensions = {".zip", ".jez", ".tmp", ".part", ".log"}
offending_files = [
    str(p.relative_to(DATASET_PATH))
    for p in Path(DATASET_PATH).rglob("*")
    if p.is_file() and any(sfx.lower() in forbidden_extensions for sfx in p.suffixes)
]
if offending_files:
    raise ValueError(
        f"Upload abortado! Arquivos brutos/temporários detectados em DATASET_PATH: {offending_files[:10]}"
    )

if is_directory_empty(Path(DATASET_PATH)):
    raise RuntimeError(
        f"Upload abortado! DATASET_PATH ({DATASET_PATH}) está vazio após o processamento."
    )

if globals().get("__name__") == "__main__" and not os.getenv("PYTEST_CURRENT_TEST"):
    import kagglehub

    print(f"Publicando nova versão do dataset '{DATASET_HANDLE}' a partir de {DATASET_PATH}...")
    kagglehub.dataset_upload(
        handle=DATASET_HANDLE,
        local_dataset_dir=str(DATASET_PATH),
        version_notes=f"[{PHASE_ID}] {PHASE_SUMMARY}",
        ignore_patterns=[
            "*.tmp",
            "*.part",
            "*.zip",
            "*.jez",
            "*.log",
            "__pycache__/",
            ".ipynb_checkpoints/",
            ".DS_Store",
        ],
    )
    print(f"[SUCESSO] Nova versão de '{DATASET_HANDLE}' publicada no Kaggle!")

## Limitações e Notas

- **Cobertura do catálogo**: O `event_catalog.yml` cobre 16 famílias de eventos oficialmente documentadas pelo TSE. Mensagens que não correspondam a nenhum padrão recebem `event_code = UNKNOWN_EVENT` e `classification_status = UNKNOWN`. Isso é esperado para linhas de diagnóstico, auditoria e operações internas não documentadas.

- **Extração de PII**: A célula de classificação não retém texto livre nem extrai dados pessoais. O único valor numérico extraído é `biometric_attempt_num` (1–127), derivado de um grupo nomeado inócuo do regex.

- **Dependência do Bronze**: Este notebook depende da existência dos Parquet Bronze em `fact_urna_logs/election_year=2026/turn=T/uf=UF/`. Se o Bronze de um turno/UF estiver ausente, a partição é silenciosamente ignorada.

- **Coluna `application`**: Se o Bronze não possuir a coluna `application`, o valor padrão `"VOTA"` é assumido para todas as linhas, o que pode afetar a filtragem por aplicativo nos padrões multi-aplicação (`GAP`, `RED`, etc.).

- **Idempotência**: A execução é idempotente — Parquet Silver existentes são sobrescritos. Use `SILVER_OUT` em diretório de trabalho (`{DATASET_PATH}/silver/`) para garantir limpeza entre runs.